# Supplementary OWID Datasets for Inequality-Democratic Resilience Robustness Checks

## Description

This notebook demonstrates the collection and formatting of supplementary panel datasets from Our World in Data (OWID) for robustness checks in inequality-democratic resilience research.

### Datasets Collected:
1. **Polity V Democracy Index** - Political regime characteristics and democracy scores (-10 to +10 scale)
2. **Oil Rents (% of GDP)** - Natural resource dependence with binary oil exporter indicator (>20% GDP threshold)
3. **Gini Coefficient (Alternative)** - Income inequality measure from World Bank Poverty and Inequality Platform

### Data Coverage:
- Time period: 1990-2022
- Format: Panel data (country-year observations)
- Output schema: JSON with `datasets` array containing `input`, `output`, and `metadata_*` fields

### Original Script:
This notebook reproduces the key processing steps from `data.py` which loads OWID datasets, formats them into a standardized schema, and outputs JSON files for use in the research pipeline.

## Install Dependencies

Install required packages. Packages pre-installed on Google Colab (numpy, pandas, etc.) are only installed locally to match Colab's environment.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru - NOT on Colab, always install
_pip('loguru==0.7.3')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

print('Dependencies installed successfully!')

## Imports

Import all required libraries for data processing and formatting.

In [ ]:
from loguru import logger
from pathlib import Path
import json
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Setup logger
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

print('Imports complete!')

## Data Loading Helper

Load demo data from GitHub URL with local fallback. This allows the notebook to work both in Colab (after deployment) and locally during development.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-f1091c-welfare-state-generosity-and-the-inequal/main/round-1/dataset-2/demo/mini_demo_data.json"

import json, os

def load_data():
    """Load demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub URL load failed: {e}")
    
    # Local fallback
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json")

print('Data loading helper defined!')

## Load Demo Data

Load the mini demo dataset containing curated examples from all three supplementary datasets.

In [ ]:
# Load the demo data
data = load_data()

# Display basic info about loaded data
print(f"Number of datasets: {len(data['datasets'])}")
for dataset in data['datasets']:
    print(f"  - {dataset['dataset']}: {len(dataset['examples'])} examples")

# Show first example from each dataset
print("\n--- Sample examples ---")
for dataset in data['datasets']:
    print(f"\nDataset: {dataset['dataset']}")
    if dataset['examples']:
        example = dataset['examples'][0]
        print(f"  Country: {example['metadata_country']}")
        print(f"  Year: {example['metadata_year']}")
        print(f"  Input: {example['input'][:80]}...")
        print(f"  Output: {example['output'][:80]}...")

## Configuration

Define tunable parameters for data processing. For this demo, we use minimum values to ensure quick execution.

- `USE_MINI_DATA`: If True, use the loaded mini demo data; if False, would load full OWID datasets
- `OIL_RENTS_THRESHOLD`: Threshold for classifying oil exporters (>20% of GDP)
- `YEAR_START`, `YEAR_END`: Date range filter (1990-2022 in original)
- `MAX_EXAMPLES_PER_DATASET`: Limit examples per dataset for demo (use None for all)

In [ ]:
# Configuration parameters
USE_MINI_DATA = True  # Use demo data (set to False to process full OWID datasets)
OIL_RENTS_THRESHOLD = 20  # Percentage of GDP threshold for oil exporter classification
YEAR_START = 1990  # Start year for filtering
YEAR_END = 2022  # End year for filtering
MAX_EXAMPLES_PER_DATASET = 3  # Limit for demo (None = no limit)

print("Configuration:")
print(f"  USE_MINI_DATA: {USE_MINI_DATA}")
print(f"  OIL_RENTS_THRESHOLD: {OIL_RENTS_THRESHOLD}%")
print(f"  Date range: {YEAR_START}-{YEAR_END}")
print(f"  MAX_EXAMPLES_PER_DATASET: {MAX_EXAMPLES_PER_DATASET}")

## Process Dataset: Polity V Democracy Index

Format the Polity V dataset which contains:
- **Input fields**: country, year, executive recruitment/constraint, political competition/regulation
- **Output fields**: polity_v_democracy score (-10 to +10), regime classification

The Polity V score ranges from -10 (full autocracy) to +10 (full democracy).

In [ ]:
# Process Polity V dataset
logger.info("Processing Polity V dataset...")

polity_examples = []

for dataset in data['datasets']:
    if dataset['dataset'] == 'polity_v':
        examples = dataset['examples']
        
        # Limit examples for demo
        if MAX_EXAMPLES_PER_DATASET is not None:
            examples = examples[:MAX_EXAMPLES_PER_DATASET]
        
        for example in examples:
            # Parse input JSON
            input_data = json.loads(example['input'])
            
            # Parse output JSON
            output_data = json.loads(example['output'])
            
            polity_examples.append({
                'country': example['metadata_country'],
                'year': example['metadata_year'],
                'polity_v_democracy': output_data.get('polity_v_democracy'),
                'polity_v_regime': output_data.get('polity_v_regime'),
                'exec_reccomp': input_data.get('exec_reccomp_polity'),
                'exec_recopen': input_data.get('exec_recopen_polity'),
                'exec_constr': input_data.get('exec_constr_polity'),
                'polpart_reg': input_data.get('polpart_reg_polity'),
                'polpart_comp': input_data.get('polpart_comp_polity')
            })
        
        break

# Convert to DataFrame for display
polity_df = pd.DataFrame(polity_examples)
print(f"Polity V dataset: {len(polity_examples)} examples")
print("\nSample data:")
display(polity_df)

## Process Dataset: Oil Rents (% of GDP)

Format the oil rents dataset from World Bank WDI:
- **Input fields**: country, year
- **Output fields**: oil_rents_pct_gdp (percentage of GDP), oil_exporter (binary indicator)

Oil exporter classification uses threshold of >20% GDP (configurable via `OIL_RENTS_THRESHOLD`).

In [ ]:
# Process Oil Rents dataset
logger.info("Processing oil rents dataset...")

oil_examples = []

for dataset in data['datasets']:
    if dataset['dataset'] == 'oil_rents':
        examples = dataset['examples']
        
        # Limit examples for demo
        if MAX_EXAMPLES_PER_DATASET is not None:
            examples = examples[:MAX_EXAMPLES_PER_DATASET]
        
        for example in examples:
            # Parse input and output JSON
            input_data = json.loads(example['input'])
            output_data = json.loads(example['output'])
            
            oil_examples.append({
                'country': example['metadata_country'],
                'year': example['metadata_year'],
                'oil_rents_pct_gdp': output_data.get('oil_rents_pct_gdp'),
                'oil_exporter': output_data.get('oil_exporter'),
                'threshold_used': OIL_RENTS_THRESHOLD
            })
        
        break

# Convert to DataFrame for display
oil_df = pd.DataFrame(oil_examples)
print(f"Oil rents dataset: {len(oil_examples)} examples")
print("\nSample data:")
display(oil_df)

## Process Dataset: Gini Coefficient (Alternative)

Format the Gini coefficient dataset from World Bank Poverty and Inequality Platform:
- **Input fields**: country, year
- **Output fields**: gini_coeff_alt (Gini coefficient, 0-1 scale)

The Gini coefficient measures income inequality: 0 = perfect equality, 1 = perfect inequality.

In [ ]:
# Process Gini Coefficient dataset
logger.info("Processing Gini coefficient dataset...")

gini_examples = []

for dataset in data['datasets']:
    if dataset['dataset'] == 'gini_coeff_alt':
        examples = dataset['examples']
        
        # Limit examples for demo
        if MAX_EXAMPLES_PER_DATASET is not None:
            examples = examples[:MAX_EXAMPLES_PER_DATASET]
        
        for example in examples:
            # Parse input and output JSON
            input_data = json.loads(example['input'])
            output_data = json.loads(example['output'])
            
            gini_examples.append({
                'country': example['metadata_country'],
                'year': example['metadata_year'],
                'gini_coeff': output_data.get('gini_coeff_alt')
            })
        
        break

# Convert to DataFrame for display
gini_df = pd.DataFrame(gini_examples)
print(f"Gini coefficient dataset: {len(gini_examples)} examples")
print("\nSample data:")
display(gini_df)

## Results Summary

Display summary statistics and visualizations for the three processed datasets.

In [ ]:
# Summary statistics
print("="*60)
print("RESULTS SUMMARY")
print("="*60)

print("\n1. Polity V Democracy Index")
print("-"*40)
if not polity_df.empty:
    print(f"   Examples: {len(polity_df)}")
    print(f"   Countries: {polity_df['country'].nunique()}")
    print(f"   Democracy score range: [{polity_df['polity_v_democracy'].min()}, {polity_df['polity_v_democracy'].max()}]")
    print(f"   Mean democracy score: {polity_df['polity_v_democracy'].mean():.2f}")

print("\n2. Oil Rents (% of GDP)")
print("-"*40)
if not oil_df.empty:
    print(f"   Examples: {len(oil_df)}")
    print(f"   Countries: {oil_df['country'].nunique()}")
    print(f"   Oil rents range: [{oil_df['oil_rents_pct_gdp'].min():.2f}%, {oil_df['oil_rents_pct_gdp'].max():.2f}%]")
    print(f"   Oil exporters: {oil_df['oil_exporter'].sum()} / {len(oil_df)}")

print("\n3. Gini Coefficient (Alternative)")
print("-"*40)
if not gini_df.empty:
    print(f"   Examples: {len(gini_df)}")
    print(f"   Countries: {gini_df['country'].nunique()}")
    print(f"   Gini range: [{gini_df['gini_coeff'].min():.3f}, {gini_df['gini_coeff'].max():.3f}]")
    print(f"   Mean Gini: {gini_df['gini_coeff'].mean():.3f}")

print("\n" + "="*60)

## Visualization

Create visualizations to compare the three datasets.

In [ ]:
# Create visualizations
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Polity V Democracy Scores
if not polity_df.empty:
    axes[0].bar(polity_df['country'], polity_df['polity_v_democracy'])
    axes[0].set_title('Polity V Democracy Scores')
    axes[0].set_xlabel('Country')
    axes[0].set_ylabel('Democracy Score (-10 to +10)')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# Plot 2: Oil Rents
if not oil_df.empty:
    colors = ['red' if e == 1 else 'blue' for e in oil_df['oil_exporter']]
    axes[1].bar(oil_df['country'], oil_df['oil_rents_pct_gdp'], color=colors)
    axes[1].set_title('Oil Rents (% of GDP)')
    axes[1].set_xlabel('Country')
    axes[1].set_ylabel('Oil Rents (% of GDP)')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].axhline(y=OIL_RENTS_THRESHOLD, color='red', linestyle='--', alpha=0.5, label=f'{OIL_RENTS_THRESHOLD}% threshold')
    axes[1].legend()

# Plot 3: Gini Coefficients
if not gini_df.empty:
    axes[2].bar(gini_df['country'], gini_df['gini_coeff'])
    axes[2].set_title('Gini Coefficient')
    axes[2].set_xlabel('Country')
    axes[2].set_ylabel('Gini Coefficient (0-1)')
    axes[2].tick_params(axis='x', rotation=45)
    axes[2].axhline(y=0.3, color='gray', linestyle='--', alpha=0.5, label='0.3 threshold')
    axes[2].legend()

plt.tight_layout()
plt.show()

# Print interpretation
print("\nInterpretation:")
print("- Polity V: Scores range from -10 (autocracy) to +10 (democracy)")
print("- Oil Rents: Red bars indicate oil exporters (>20% GDP)")
print("- Gini: Higher values indicate greater income inequality")

## Demo Complete!

This notebook demonstrated the key processing steps for formatting supplementary OWID datasets:

1. **Data Loading**: Loaded curated demo data from GitHub URL with local fallback
2. **Data Processing**: Formatted each dataset into standardized schema
3. **Results**: Displayed summary statistics and visualizations

### To run with full datasets:
- Set `USE_MINI_DATA = False` in the Configuration cell
- Update file paths to point to full OWID dataset files
- Increase `MAX_EXAMPLES_PER_DATASET` to `None` to process all examples

### Output Files (from original script):
- `full_data_out.json`: Complete dataset (5.9 MB)
- `mini_data_out.json`: Mini dataset (3 examples per dataset)
- `preview_data_out.json`: Preview dataset (truncated fields)